# Fine-Tuning Indic-Speak on Full Marathi Dataset (100% Anagha Corpus)

This standalone notebook runs LoRA fine-tuning across the **entire filtered Anagha dataset (~6,829 utterances)** on a Kaggle NVIDIA T4 GPU.
- **Base Model**: `bodhan-ai/indic-speak` (3.3B parameters, LLaMA-3.2 backbone)
- **Target Modules**: Attention (`q, k, v, o`) + SwiGLU MLP (`gate, up, down`)
- **Dataset**: `snorbyte/indic-tts-sample-snac-encoded` (100% Anagha corpus, ~6,829 samples)
- **Epochs**: 2 full epochs (~1,708 optimizer steps)
- **Output Directory**: `outputs/lora_marathi_full/`

In [ ]:
# 1. Clone repository and install dependencies
import os
repo_dir = "/kaggle/working/indic-speak-marathi-finetune-full"
if not os.path.exists(repo_dir):
    !git clone https://github.com/mehersoni/indic-speak-marathi-finetune.git {repo_dir}
%cd {repo_dir}
!git pull

# Remove conflicting pre-installed torchao on Kaggle and install dependencies
!pip uninstall -y torchao
!pip install -q "transformers>=5" peft accelerate snac soundfile datasets pandas pyarrow huggingface_hub pyyaml

In [ ]:
# 2. Hugging Face Authentication
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face authenticated successfully.")
else:
    print("HF_TOKEN not detected in Kaggle Secrets. If needed, set os.environ['HF_TOKEN'] = 'your_token'.")

In [ ]:
# 3. Prepare and filter Marathi dataset (Threshold: 1400 tokens)
!CUDA_VISIBLE_DEVICES=0 python scripts/prepare_dataset.py

In [ ]:
# 4. Run GPU Smoke Test with Full Dataset Config
!CUDA_VISIBLE_DEVICES=0 python scripts/smoke_test.py --config configs/marathi_lora_full.yaml

In [ ]:
# 5. Train LoRA adapter on Full Marathi Anagha dataset (~6,829 samples)
!CUDA_VISIBLE_DEVICES=0 python src/train.py --config configs/marathi_lora_full.yaml

In [ ]:
# 6. Pull latest inference code with loop-prevention fix
!git pull

# Generate speech samples with Base Model (Baseline comparison)
!CUDA_VISIBLE_DEVICES=0 python src/inference.py \
    --model bodhan-ai/indic-speak \
    --out-dir outputs/examples_full

In [ ]:
# 7. Generate speech samples with Full-Dataset LoRA Adapter (WITH LOOP FIX)
# src/inference.py uses adaptive_max = min(max_new_tokens, max(280, len(text)*14)) + repetition_penalty=1.1
!CUDA_VISIBLE_DEVICES=0 python src/inference.py \
    --model bodhan-ai/indic-speak \
    --adapter outputs/lora_marathi_full/final_adapter \
    --out-dir outputs/examples_full

In [ ]:
# 8. Audio Diagnostics & Playback Verification
import soundfile as sf
import numpy as np
from pathlib import Path
import IPython.display as ipd

audio_dir = Path("outputs/examples_full")
wav_files = sorted(audio_dir.glob("*.wav"))

print(f"=== Run 3 Audio Quality & Duration Diagnostics ({len(wav_files)} files) ===")
print(f"{'Filename':<22} | {'Duration':>8} | {'RMS Energy':>10} | {'Peak Amp':>8} | {'Loop Check'}")
print("-" * 72)

for wf in wav_files:
    wav, sr = sf.read(str(wf))
    dur = len(wav) / sr
    rms = float(np.sqrt(np.mean(wav**2)))
    peak = float(np.max(np.abs(wav)))
    loop_status = "PASS (Normal)" if dur < 10.0 else "FAIL (Looping)"
    print(f"{wf.name:<22} | {dur:>7.2f}s | {rms:>10.4f} | {peak:>8.4f} | {loop_status}")

print("\n--- Audio Playback ---")
for wf in wav_files:
    if wf.name.startswith("finetuned"):
        print(f"▶ Playing: {wf.name}")
        ipd.display(ipd.Audio(str(wf)))

In [ ]:
# 9. Package Full Run 3 Submission as ZIP for Output Tab Download
import os, shutil

WORKING = "/kaggle/working"
SUBMISSION_DIR = f"{WORKING}/marathi_tts_run3_submission"
os.makedirs(f"{SUBMISSION_DIR}/audio/base", exist_ok=True)
os.makedirs(f"{SUBMISSION_DIR}/audio/finetuned", exist_ok=True)
os.makedirs(f"{SUBMISSION_DIR}/model/final_adapter", exist_ok=True)

# 1. Copy synthesized audio
if os.path.exists("outputs/examples_full"):
    for f in os.listdir("outputs/examples_full"):
        if f.endswith(".wav"):
            if f.startswith("base"):
                shutil.copy(f"outputs/examples_full/{f}", f"{SUBMISSION_DIR}/audio/base/{f}")
            else:
                shutil.copy(f"outputs/examples_full/{f}", f"{SUBMISSION_DIR}/audio/finetuned/{f}")

# 2. Copy trained LoRA adapter weights & configs
ADAPTER_DIR = "outputs/lora_marathi_full/final_adapter"
if os.path.exists(ADAPTER_DIR):
    shutil.copytree(ADAPTER_DIR, f"{SUBMISSION_DIR}/model/final_adapter", dirs_exist_ok=True)

# 3. Copy trainer_state.json for full step-by-step loss history
STATE_FILE = "outputs/lora_marathi_full/trainer_state.json"
if os.path.exists(STATE_FILE):
    shutil.copy(STATE_FILE, f"{SUBMISSION_DIR}/trainer_state.json")

# 4. Zip entire submission folder into /kaggle/working
zip_path = f"{WORKING}/marathi_tts_run3_submission"
shutil.make_archive(zip_path, "zip", SUBMISSION_DIR)

print(f"\n✓ Complete Run 3 submission packaged successfully!")
print(f"  Archive: {zip_path}.zip")
print(f"  Size:    {os.path.getsize(f'{zip_path}.zip') / (1024*1024):.2f} MB")
print("\n→ Go to the right sidebar in Kaggle, expand the 'Output' tab, and click Download on marathi_tts_run3_submission.zip.")